# Практическое занятие 1: Сбор данных с hh.ru API

**Цель:** Изучить процесс сбора данных для задач NLP с использованием асинхронности.

**Задачи:**
1. Асинхронный сбор вакансий с hh.ru
2. Предварительная обработка текстовых данных
3. Сохранение датасета

## 1. Импорт библиотек

In [3]:
import aiohttp
import asyncio
import pandas as pd
import re
import time
import random
from bs4 import BeautifulSoup
from tqdm.asyncio import tqdm_asyncio

# Для работы asyncio в Jupyter
import nest_asyncio
nest_asyncio.apply()

## 2. Настройки API hh.ru

API hh.ru бесплатный и не требует ключа для базовых запросов.

**Основные endpoints:**
- `GET /vacancies` - поиск вакансий
- `GET /vacancies/{id}` - детальная информация о вакансии

In [4]:
# Базовый URL API
BASE_URL = "https://api.hh.ru"

# Заголовки для запросов (User-Agent обязателен)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json",
    "Accept-Language": "ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7"
}

# Параметры поиска
SEARCH_QUERY = "ML"  # Поисковый запрос
AREA_ID = 123  # 1 = Москва, 2 = Санкт-Петербург (можно убрать для всей России)
PER_PAGE = 100  # Вакансий на страницу (максимум 100)
MAX_PAGES = 20  # Максимум страниц для сбора (API ограничивает 2000 вакансий)

# Настройки для обхода rate limiting
CONCURRENT_REQUESTS = 2  # Уменьшили до 2 одновременных запросов
MAX_RETRIES = 3  # Количество повторных попыток при ошибке
BASE_DELAY = 0.5  # Базовая задержка между запросами (секунды)

## 3. Асинхронный сбор данных

In [5]:
async def fetch_vacancy_ids(session, query, area, page, per_page):
    """
    Получение списка ID вакансий с одной страницы поиска.
    
    Args:
        session: aiohttp сессия
        query: поисковый запрос
        area: ID региона
        page: номер страницы
        per_page: количество на странице
    
    Returns:
        список ID вакансий и общее количество найденных
    """
    params = {
        "text": query,
        "area": area,
        "page": page,
        "per_page": per_page
    }
    
    try:
        async with session.get(f"{BASE_URL}/vacancies", params=params) as response:
            if response.status == 200:
                data = await response.json()
                vacancy_ids = [item["id"] for item in data.get("items", [])]
                total_found = data.get("found", 0)
                return vacancy_ids, total_found
            else:
                print(f"Ошибка {response.status} на странице {page}")
                return [], 0
    except Exception as e:
        print(f"Ошибка при получении страницы {page}: {e}")
        return [], 0

In [6]:
async def fetch_vacancy_details(session, vacancy_id, semaphore):
    """
    Получение детальной информации о вакансии с retry при ошибках.
    Используем семафор для ограничения одновременных запросов.
    
    Args:
        session: aiohttp сессия
        vacancy_id: ID вакансии
        semaphore: семафор для ограничения параллелизма
    
    Returns:
        словарь с данными вакансии или None
    """
    async with semaphore:  # Ограничиваем количество одновременных запросов
        
        for attempt in range(MAX_RETRIES):
            try:
                # Случайная задержка перед запросом (0.3-0.8 сек)
                await asyncio.sleep(BASE_DELAY + random.uniform(0, 0.3))
                
                async with session.get(f"{BASE_URL}/vacancies/{vacancy_id}") as response:
                    if response.status == 200:
                        data = await response.json()
                        
                        # Извлекаем нужные поля
                        salary = data.get("salary")
                        employer = data.get("employer", {})
                        
                        return {
                            "id": vacancy_id,
                            "name": data.get("name", ""),
                            "description": data.get("description", ""),  # HTML описание
                            "salary_from": salary.get("from") if salary else None,
                            "salary_to": salary.get("to") if salary else None,
                            "salary_currency": salary.get("currency") if salary else None,
                            "salary_gross": salary.get("gross") if salary else None,
                            "employer_id": employer.get("id"),
                            "employer_name": employer.get("name", ""),
                            "area": data.get("area", {}).get("name", ""),
                            "experience": data.get("experience", {}).get("name", ""),
                            "employment": data.get("employment", {}).get("name", ""),
                            "schedule": data.get("schedule", {}).get("name", ""),
                            "key_skills": ", ".join([s["name"] for s in data.get("key_skills", [])]),
                            "url": data.get("alternate_url", "")
                        }
                    
                    elif response.status == 404:
                        return None  # Вакансия удалена
                    
                    elif response.status == 403:
                        # Rate limit - ждем и пробуем снова
                        wait_time = (2 ** attempt) + random.uniform(1, 3)
                        await asyncio.sleep(wait_time)
                        continue
                    
                    else:
                        return None
                        
            except Exception as e:
                if attempt < MAX_RETRIES - 1:
                    await asyncio.sleep(2 ** attempt)
                    continue
                return None
        
        return None  # Все попытки исчерпаны

In [7]:
async def collect_vacancies(query, area, max_pages, per_page, concurrent_limit):
    """
    Основная функция сбора вакансий с обработкой батчами.
    
    1. Сначала собираем все ID вакансий со страниц поиска
    2. Затем асинхронно получаем детали каждой вакансии батчами
    
    Args:
        query: поисковый запрос
        area: ID региона
        max_pages: максимум страниц
        per_page: вакансий на страницу
        concurrent_limit: лимит одновременных запросов
    
    Returns:
        список словарей с данными вакансий
    """
    all_vacancy_ids = []
    
    # Настройки для батчевой обработки
    BATCH_SIZE = 50  # Обрабатываем по 50 вакансий за раз
    BATCH_DELAY = 5  # Пауза между батчами (секунды)
    
    async with aiohttp.ClientSession(headers=HEADERS) as session:
        # Шаг 1: Собираем ID вакансий со всех страниц
        print("Сбор ID вакансий...")
        
        # Сначала узнаем сколько всего вакансий
        first_ids, total_found = await fetch_vacancy_ids(session, query, area, 0, per_page)
        all_vacancy_ids.extend(first_ids)
        
        print(f"Найдено вакансий: {total_found}")
        
        # Определяем сколько страниц нужно собрать
        total_pages = min(max_pages, (total_found // per_page) + 1)
        print(f"Будет собрано страниц: {total_pages}")
        
        # Собираем остальные страницы с задержкой
        for page in range(1, total_pages):
            ids, _ = await fetch_vacancy_ids(session, query, area, page, per_page)
            all_vacancy_ids.extend(ids)
            await asyncio.sleep(0.5)  # Задержка между страницами
        
        # Убираем дубликаты
        all_vacancy_ids = list(set(all_vacancy_ids))
        print(f"Уникальных вакансий для сбора: {len(all_vacancy_ids)}")
        
        # Шаг 2: Собираем детали вакансий батчами
        print(f"\nСбор детальной информации батчами по {BATCH_SIZE} вакансий...")
        print(f"Пауза между батчами: {BATCH_DELAY} сек")
        
        semaphore = asyncio.Semaphore(concurrent_limit)
        all_vacancies = []
        
        # Разбиваем на батчи
        for i in range(0, len(all_vacancy_ids), BATCH_SIZE):
            batch_ids = all_vacancy_ids[i:i + BATCH_SIZE]
            batch_num = i // BATCH_SIZE + 1
            total_batches = (len(all_vacancy_ids) + BATCH_SIZE - 1) // BATCH_SIZE
            
            print(f"\nБатч {batch_num}/{total_batches} ({len(batch_ids)} вакансий)...")
            
            # Создаем задачи для текущего батча
            tasks = [
                fetch_vacancy_details(session, vid, semaphore) 
                for vid in batch_ids
            ]
            
            # Выполняем батч
            results = await tqdm_asyncio.gather(*tasks, desc=f"Батч {batch_num}")
            
            # Добавляем успешные результаты
            successful = [v for v in results if v is not None]
            all_vacancies.extend(successful)
            print(f"Успешно: {len(successful)}/{len(batch_ids)}")
            
            # Пауза между батчами (кроме последнего)
            if i + BATCH_SIZE < len(all_vacancy_ids):
                print(f"Пауза {BATCH_DELAY} сек перед следующим батчем...")
                await asyncio.sleep(BATCH_DELAY)
        
        print(f"\n{'='*50}")
        print(f"Всего успешно собрано: {len(all_vacancies)} из {len(all_vacancy_ids)}")
        
        return all_vacancies

In [8]:
# Запускаем сбор данных
start_time = time.time()

vacancies = asyncio.run(
    collect_vacancies(
        query=SEARCH_QUERY,
        area=AREA_ID,
        max_pages=MAX_PAGES,
        per_page=PER_PAGE,
        concurrent_limit=CONCURRENT_REQUESTS
    )
)

elapsed_time = time.time() - start_time
print(f"\nВремя сбора: {elapsed_time:.2f} секунд")

Сбор ID вакансий...
Найдено вакансий: 9
Будет собрано страниц: 1
Уникальных вакансий для сбора: 9

Сбор детальной информации батчами по 50 вакансий...
Пауза между батчами: 5 сек

Батч 1/1 (9 вакансий)...


Батч 1: 100%|██████████| 9/9 [00:04<00:00,  2.03it/s]

Успешно: 9/9

Всего успешно собрано: 9 из 9

Время сбора: 4.81 секунд


In [9]:
# Создаем DataFrame
df = pd.DataFrame(vacancies)
print(f"Размер датасета: {df.shape}")
df.head()

Размер датасета: (9, 15)


,id,name,description,salary_from,salary_to,salary_currency,salary_gross,employer_id,employer_name,area,experience,employment,schedule,key_skills,url
0,127996365,Главный кредитный аналитик,<p><strong>Обязанности:</strong></p> <ul> <li>...,NaN,NaN,None,None,4181,Банк ВТБ (ПАО),Луганск,От 1 года до 3 лет,Полная занятость,Полный день,"Финансовый анализ, Анализ ФХД, Оценка рисков, ...",https://hh.ru/vacancy/127996365
1,128695630,Руководитель группы бизнес-анализа,<p><strong>В ООО &quot;Луганская телефонная ко...,NaN,NaN,None,None,2003415,Миранда-медиа,Луганск,От 1 года до 3 лет,Полная занятость,Полный день,"Анализ бизнес-процессов, Оптимизация бизнес-пр...",https://hh.ru/vacancy/128695630
2,128744660,Кладовщик-комплектовщик,"<p> </p> <p><em><strong>Крупная сеть СТО,склад...",60000.0,80000.0,RUR,True,10253840,АвтоСила (ООО Автомотозапчасть),Луганск,Нет опыта,Полная занятость,Полный день,,https://hh.ru/vacancy/128744660
3,124818550,Специалист по охране труда (розничная сеть),<p><strong>АЙСИ ГРУПП</strong> -крупный межотр...,NaN,NaN,None,None,1898133,АйСи Инвест,Луганск,От 3 до 6 лет,Полная занятость,Полный день,,https://hh.ru/vacancy/124818550
4,128543777,Менеджер по работе с ключевыми клиентами (КАМ),<p>Крупная коммерческая компания в поиске &quo...,NaN,NaN,None,None,10695925,Калипсо,Луганск,От 1 года до 3 лет,Полная занятость,Полный день,"Поиск и привлечение клиентов, B2B Продажи, Раз...",https://hh.ru/vacancy/128543777


## 4. Предварительная обработка текста

In [10]:
def clean_html(html_text):
    """
    Очистка текста от HTML тегов.
    
    Args:
        html_text: текст с HTML разметкой
    
    Returns:
        очищенный текст
    """
    if not html_text:
        return ""
    
    # Парсим HTML
    soup = BeautifulSoup(html_text, "html.parser")
    
    # Извлекаем текст
    text = soup.get_text(separator=" ")
    
    return text

In [11]:
def preprocess_text(text):
    """
    Предобработка текста:
    - удаление лишних пробелов
    - удаление спецсимволов
    - приведение к нижнему регистру
    
    Args:
        text: исходный текст
    
    Returns:
        обработанный текст
    """
    if not text:
        return ""
    
    # Приводим к нижнему регистру
    text = text.lower()
    
    # Удаляем URL
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Удаляем email
    text = re.sub(r'\S+@\S+', '', text)
    
    # Оставляем только буквы (кириллица + латиница), цифры и пробелы
    text = re.sub(r'[^а-яёa-z0-9\s]', ' ', text)
    
    # Удаляем множественные пробелы
    text = re.sub(r'\s+', ' ', text)
    
    # Удаляем пробелы в начале и конце
    text = text.strip()
    
    return text

In [12]:
# Применяем предобработку к описаниям вакансий
print("Очистка HTML...")
df["description_clean"] = df["description"].apply(clean_html)

print("Предобработка текста...")
df["description_processed"] = df["description_clean"].apply(preprocess_text)

print("Готово!")

Очистка HTML...
Предобработка текста...
Готово!


In [13]:
# Посмотрим на результат предобработки
print("Пример исходного описания (HTML):")
print(df["description"].iloc[0][:500])
print("\n" + "="*50 + "\n")
print("Пример обработанного описания:")
print(df["description_processed"].iloc[0][:500])

Пример исходного описания (HTML):
<p><strong>Обязанности:</strong></p> <ul> <li>анализ финансово-хозяйственной деятельности (АФХД) и платежеспособности компаний заёмщиков сегмента до 1 млрд.;</li> <li>проведение оценки рисков по итогам АФХД;</li> <li>участие в Кредитных Комитетах Банка;</li> <li>подготовка предложений клиентам по условиям кредитования;</li> <li>составление резюме по кредитной сделке;</li> <li>взаимодействие с клиентами на всех этапах рассмотрения сделки;</li> <li>подача/получение документов в Государственных Орг


Пример обработанного описания:
обязанности анализ финансово хозяйственной деятельности афхд и платежеспособности компаний заёмщиков сегмента до 1 млрд проведение оценки рисков по итогам афхд участие в кредитных комитетах банка подготовка предложений клиентам по условиям кредитования составление резюме по кредитной сделке взаимодействие с клиентами на всех этапах рассмотрения сделки подача получение документов в государственных органах например при регистрации

In [14]:
# Статистика по датасету
print("Статистика датасета:")
print(f"Всего вакансий: {len(df)}")
print(f"Вакансий с зарплатой: {df['salary_from'].notna().sum()}")
print(f"Уникальных работодателей: {df['employer_name'].nunique()}")
print(f"Средняя длина описания: {df['description_processed'].str.len().mean():.0f} символов")

Статистика датасета:
Всего вакансий: 9
Вакансий с зарплатой: 3
Уникальных работодателей: 8
Средняя длина описания: 1418 символов


## 5. Сохранение датасета

In [15]:
# Сохраняем в CSV
csv_path = "vacancies_dataset.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"Датасет сохранен в {csv_path}")

# Сохраняем в JSON (удобно для сложных структур)
json_path = "vacancies_dataset.json"
df.to_json(json_path, orient="records", force_ascii=False, indent=2)
print(f"Датасет сохранен в {json_path}")

Датасет сохранен в vacancies_dataset.csv
Датасет сохранен в vacancies_dataset.json


In [16]:
# Проверяем что датасет можно загрузить
df_loaded = pd.read_csv(csv_path)
print(f"Загружено из CSV: {len(df_loaded)} записей")
df_loaded.head(3)

Загружено из CSV: 9 записей


,id,name,description,salary_from,salary_to,salary_currency,salary_gross,employer_id,employer_name,area,experience,employment,schedule,key_skills,url,description_clean,description_processed
0,127996365,Главный кредитный аналитик,<p><strong>Обязанности:</strong></p> <ul> <li>...,NaN,NaN,NaN,NaN,4181,Банк ВТБ (ПАО),Луганск,От 1 года до 3 лет,Полная занятость,Полный день,"Финансовый анализ, Анализ ФХД, Оценка рисков, ...",https://hh.ru/vacancy/127996365,Обязанности: анализ финансово-хозяйственно...,обязанности анализ финансово хозяйственной дея...
1,128695630,Руководитель группы бизнес-анализа,<p><strong>В ООО &quot;Луганская телефонная ко...,NaN,NaN,NaN,NaN,2003415,Миранда-медиа,Луганск,От 1 года до 3 лет,Полная занятость,Полный день,"Анализ бизнес-процессов, Оптимизация бизнес-пр...",https://hh.ru/vacancy/128695630,"В ООО ""Луганская телефонная компания"" (ГК ""Мир...",в ооо луганская телефонная компания гк миранда...
2,128744660,Кладовщик-комплектовщик,"<p> </p> <p><em><strong>Крупная сеть СТО,склад...",60000.0,80000.0,RUR,True,10253840,АвтоСила (ООО Автомотозапчасть),Луганск,Нет опыта,Полная занятость,Полный день,NaN,https://hh.ru/vacancy/128744660,"Крупная сеть СТО,складов автозапчастей и т...",крупная сеть сто складов автозапчастей и точек...


## 6. Анализ собранных данных

In [17]:
# Распределение по опыту работы
print("Распределение по опыту:")
print(df["experience"].value_counts())

Распределение по опыту:
experience
От 1 года до 3 лет    6
От 3 до 6 лет         2
Нет опыта             1
Name: count, dtype: int64


In [18]:
# Топ-10 работодателей
print("Топ-10 работодателей:")
print(df["employer_name"].value_counts().head(10))

Топ-10 работодателей:
employer_name
Банк ВТБ (ПАО)                     2
Миранда-медиа                      1
АвтоСила (ООО Автомотозапчасть)    1
АйСи Инвест                        1
Калипсо                            1
НПК ЛЭМЗ-ОГМК                      1
СБЕР                               1
ППК Фонд развития территорий       1
Name: count, dtype: int64


In [19]:
# Статистика по зарплатам (только RUR)
df_rur = df[df["salary_currency"] == "RUR"].copy()
if len(df_rur) > 0:
    print(f"Вакансий с зарплатой в рублях: {len(df_rur)}")
    print(f"\nЗарплата 'от' (RUR):")
    print(df_rur["salary_from"].describe())
else:
    print("Нет вакансий с указанной зарплатой в рублях")

Вакансий с зарплатой в рублях: 3

Зарплата 'от' (RUR):
count         3.000000
mean      94666.666667
std       31069.813861
min       60000.000000
25%       82000.000000
50%      104000.000000
75%      112000.000000
max      120000.000000
Name: salary_from, dtype: float64


In [20]:
# Самые частые ключевые навыки
from collections import Counter

all_skills = []
for skills in df["key_skills"].dropna():
    if skills:
        all_skills.extend([s.strip() for s in skills.split(",")])

skill_counts = Counter(all_skills)
print("Топ-15 ключевых навыков:")
for skill, count in skill_counts.most_common(15):
    print(f"  {skill}: {count}")

Топ-15 ключевых навыков:
  Аналитическое мышление: 2
  Финансовый анализ: 1
  Анализ ФХД: 1
  Оценка рисков: 1
  Консолидация финансовой отчетности: 1
  Финансовая отчетность: 1
  Анализ бизнес-процессов: 1
  Оптимизация бизнес-процессов: 1
  Бизнес-анализ: 1
  Стратегическое планирование: 1
  Организация работы сотрудников: 1
  Анализ требований: 1
  Автоматизация бизнес-процессов: 1
  Управление бизнес процессами: 1
  Поиск и привлечение клиентов: 1


## Итоги

В данной работе было выполнено:
1. **Асинхронный сбор данных** с hh.ru API с использованием `aiohttp` и `asyncio`
2. **Ограничение параллелизма** через семафор для предотвращения перегрузки сервера
3. **Предварительная обработка текста**: очистка HTML, удаление спецсимволов, нормализация
4. **Сохранение датасета** в форматах CSV и JSON

Асинхронный подход позволяет значительно ускорить сбор данных по сравнению с последовательными запросами.